# Friday practice: Can a bigger cache fix this audio job?
IT 520 · October 30, 2026 · Individual, virtual practice · About 50 minutes

An audio workstation repeatedly reads a few **control settings** while scanning fresh
**audio data**. The settings are used again; each scanned block is used once.
Can you reduce trips to main memory by changing cache capacity, the order of reads, or both?
All addresses and workloads here are invented teaching data. No audio knowledge is needed.

**Start (5 minutes):** File → Save a copy in Drive. Put your name in its title.
Run cells in order using ▶. Do not Run all: write each prediction before the
experiment it predicts. Double-click an answer cell to replace `[replace]`;
Shift+Enter saves it. Keep the outputs in your copy.

You edit settings, not the simulator. If disconnected, reconnect and rerun earlier
code cells in order. Every trial starts empty, so earlier trials cannot warm it.


## The model (8 minutes, including Q1)
A **cache** holds copies near the CPU. A **hit** finds a requested byte in the cache.
A **miss** fetches its whole block from main memory. Here every block is **16 bytes**:
block 0 is addresses 0–15, block 1 is 16–31, and so on. Thus
`block number = byte address // 16` (whole-number division).

Capacity is the number of blocks that fit. Any block may occupy any slot.
On a miss when full, discard the block **used least recently** (LRU).
A hit makes that block most recently used too. There are no writes, prefetches,
other programs, multiple cache levels or address translation in this model.

**Worked example:** capacity 1, reads at byte addresses 0, 4, 16.
0 misses and loads block 0; 4 hits because the *whole* block arrived;
16 misses and replaces block 0 with block 1.
Using a nearby byte is **spatial locality**; using the same data again soon is
**temporal locality**. Nearby data helps only if it is still cached when needed.

**Q1 — Before running:** Start empty with capacity **2 blocks**. For addresses
`0, 4, 16, 0, 32, 16`, write H or M for each read. Which block does the read
at 32 discard? Explain how the second read at 0 affects that choice.


**Q1 prediction:** [replace]

**Q1 explanation:** [replace]

In [ ]:
from collections import OrderedDict, Counter

def simulate(addresses, capacity, detail=False):
    """Cold, fully associative, read-only LRU cache; 16 bytes per block."""
    if type(capacity) is not int or capacity < 1:
        raise ValueError("capacity must be a positive whole number of blocks")
    cache = OrderedDict()
    hits = 0
    rows = []
    for address in addresses:
        if type(address) is not int or address < 0:
            raise ValueError("addresses must be nonnegative whole byte addresses")
        block = address // 16
        hit = block in cache
        hits += hit
        evicted = "-"
        if hit:
            cache.move_to_end(block)
        else:
            if len(cache) == capacity:
                evicted, _ = cache.popitem(last=False)
            cache[block] = None
        rows.append((address, block, "H" if hit else "M", evicted, list(cache)))
    result = dict(reads=len(rows), hits=hits, misses=len(rows)-hits,
                  hit_rate=100*hits/len(rows) if rows else 0)
    if detail:
        print("byte address | block | H/M | dropped block | cache oldest -> newest")
        for row in rows:
            print(row)
    return result

def workload(order="mixed", hot_blocks=3):
    if order not in ("mixed", "grouped"):
        raise ValueError('order must be "mixed" or "grouped"')
    if type(hot_blocks) is not int or not 2 <= hot_blocks <= 6:
        raise ValueError("hot_blocks must be a whole number from 2 through 6")
    # Four batches. Each batch reads every control four times, then 8 new blocks.
    controls = [16*b for b in range(hot_blocks)] * 4
    scans = [[16*b for b in range(8+8*i, 16+8*i)] for i in range(4)]
    if order == "mixed":
        return [a for scan in scans for a in controls + scan]
    return controls * 4 + [a for scan in scans for a in scan]

def trial(capacity, order="mixed", hot_blocks=3):
    result = simulate(workload(order, hot_blocks), capacity)
    print(f"{order:7} hot={hot_blocks} cache={capacity} blocks ({16*capacity} B): "
          f"{result['reads']} reads, {result['hits']} hits, {result['misses']} misses, "
          f"hit rate {result['hit_rate']:.2f}%")
    return result

print("Simulator ready. Each trial starts with an empty cache.")


In [ ]:
trace_result = simulate([0, 4, 16, 0, 32, 16], capacity=2, detail=True)
print(trace_result)

**Q1 after running:** Correct any mismatch in your trace and explain it. [replace]

## Experiment 1 — Change capacity (10 minutes)
The workstation has **3 control blocks** (blocks 0, 1, 2). One batch reads
`0, 1, 2` four times, then scans **8 new audio blocks**. There are **4 batches**;
each scan uses different blocks, so none of those audio blocks is requested again.
These are block numbers; the code converts them into byte addresses.

**Q2 — Predict:** Will going from 2 to 3 cache blocks help? Will going from 3
to 4 help as much? Why might a scan evict settings that the next batch needs?
Write this before running the capacity sweep.


**Q2 prediction and reason:** [replace]

In [ ]:
for capacity in [2, 3, 4, 12]:
    trial(capacity, order="mixed")

**Q3 — Record and explain:** Copy the four rows' hits, misses and hit rates.
Explain (a) the change from 2 to 3 and (b) what happened from 3 to 4.
Why can 12 retain controls across the scan? Use the working set:
3 control blocks plus 8 audio blocks read between control bursts.
Does a 12-block cache eliminate *all* misses? Explain using the first reads.

**Results:** [replace]

**Explanation:** [replace]


## Experiment 2 — Change order, keep the work (10 minutes)
Suppose this is an **offline** analysis job: we may do all control reads together,
then scan the audio. No result depends on interleaving these reads in this invented job.
In `grouped` order the exact same addresses occur the same number of times;
only their order changes. Both trials below use capacity **3 blocks**.

**Q4 — Predict:** Which order will miss less, and why? What is held constant?


**Q4 prediction and controlled comparison:** [replace]

In [ ]:
assert Counter(workload("mixed")) == Counter(workload("grouped"))
mixed = trial(3, "mixed")
grouped = trial(3, "grouped")
print("Misses avoided by regrouping:", mixed["misses"] - grouped["misses"])

**Q5 — Explain:** Record both hit rates and the number of misses avoided.
Why does grouping help temporal locality? Explain why there are still audio misses.
A live audio processor must read updated controls between batches; would our
regrouping automatically be valid there? State the assumption you would check.

**Results and explanation:** [replace]

**Live-processing caveat:** [replace]


## Your experiment — Move the boundary (8 minutes)
Change **only** the number of control blocks first. Pick **4, 5 or 6** below.
Keep `order="mixed"`. Your chosen control count is `H`.

**Q6 — Predict first:** At capacities `H-1` and `H`, how will the hit counts
compare? Explain what can stay in the cache during the repeated control reads.
Then run, record both outputs and explain any surprise. Do not change the simulator.


**Q6 chosen H and prediction:** [replace]

In [ ]:
HOT_BLOCKS = 4  # Change to 4, 5, or 6; keep the same value for both trials.
if HOT_BLOCKS not in [4, 5, 6]:
    raise ValueError("For this experiment choose 4, 5, or 6 control blocks.")
for capacity in [HOT_BLOCKS - 1, HOT_BLOCKS]:
    trial(capacity, "mixed", hot_blocks=HOT_BLOCKS)

**Q6 results and explanation:** [replace]

## Recommend and submit (9 minutes)
Return to the original **3 control blocks**. You can afford at most **64 bytes**
of cache data, in whole 16-byte blocks (ignore metadata). This is the **offline**
job, so either read order is allowed. Choose the **smallest cache** that gets
at least a **50% hit rate**. Use earlier evidence, then verify below.

**Q7 — Before the decision run:** State your capacity in blocks and bytes,
your order, and why you think the combination meets both constraints.
Edit the two settings below to test your recommendation.


**Q7 recommendation and prediction:** [replace]

In [ ]:
DECISION_CAPACITY = 2  # Edit: a positive whole number of blocks
DECISION_ORDER = "mixed"  # Edit: "mixed" or "grouped"
result = trial(DECISION_CAPACITY, DECISION_ORDER, hot_blocks=3)
print("Within 64-byte budget:", DECISION_CAPACITY * 16 <= 64)
print("At least 50% hits:", result["hit_rate"] >= 50)

**Q7 final evidence:** Record the final output and justify why a smaller
cache cannot meet the target in either tested order. [replace]

**Q8 — What did we establish?** Write one sentence explaining why “more cache
always gives more hits” is too strong. Then state one reason this experiment
cannot tell you how many seconds a real audio application will take. [replace]

**Submit:** Check Q1–Q8, your predictions, both personal-experiment outputs and
successful final decision output are saved. File → Save. Share → General access →
Anyone with the link → Viewer (or share directly with your instructor if that
setting is unavailable). Submit **your copy's link** in Canvas by its listed
deadline. Do not submit the starter link. No need to upload audio or other files.
